# Load upgrade0.parquet

Read the parquet file into a pandas DataFrame named `df`.

In [72]:
import pandas as pd

parquet_path = "/Users/ewilson/Documents/code/buildstock-downloader/annual/resstock_amy2018_release_1 annual/upgrade0.parquet"

df = pd.read_parquet(parquet_path)
print(df.shape)
df.head()

(549971, 771)


,bldg_id,completed_status,upgrade,in.upgrade_name,weight,applicability,in.sqft..ft2,in.representative_income,in.county_name,in.ahs_region,...,calc.weighted.emissions_reduction.electricity.lrmer_high_re_cost_15..co2e_mmt,calc.weighted.emissions_reduction.electricity.lrmer_high_re_cost_25..co2e_mmt,calc.weighted.emissions_reduction.electricity.lrmer_high_re_cost_low_ng_price_15..co2e_mmt,calc.weighted.emissions_reduction.electricity.lrmer_high_re_cost_low_ng_price_25..co2e_mmt,calc.weighted.emissions_reduction.electricity.lrmer_low_re_cost_15..co2e_mmt,calc.weighted.emissions_reduction.electricity.lrmer_low_re_cost_25..co2e_mmt,calc.weighted.emissions_reduction.electricity.lrmer_low_re_cost_high_ng_price_15..co2e_mmt,calc.weighted.emissions_reduction.electricity.lrmer_low_re_cost_high_ng_price_25..co2e_mmt,calc.weighted.emissions_reduction.electricity.lrmer_mid_case_15..co2e_mmt,calc.weighted.emissions_reduction.electricity.lrmer_mid_case_25..co2e_mmt
0,1,Success,0,Baseline,253.903673,True,1228.0,53672.0,Franklin County,Non-CBSA East South Central,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,Success,0,Baseline,253.903673,True,1138.0,21092.0,New Haven County,Non-CBSA New England,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,Success,0,Baseline,253.903673,True,854.0,25132.0,Middlesex County,"CBSA Boston-Cambridge-Newton, MA-NH",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,Success,0,Baseline,253.903673,True,854.0,43219.0,Lubbock County,Non-CBSA West South Central,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,Success,0,Baseline,253.903673,True,623.0,6051.0,Riverside County,"CBSA Riverside-San Bernardino-Ontario, CA",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [73]:
# Keep bldgid, county, state, and puma using actual parquet column names
actual_cols = ["bldg_id", "in.county", "in.state", "in.puma", "weight"]
missing_cols = [c for c in actual_cols if c not in df.columns]

if missing_cols:
    raise KeyError(f"Missing expected columns: {missing_cols}")

selected_df = df[actual_cols].copy()
selected_df.columns = ["bldgid", "county", "state", "puma", "weight"]

print("Filtered df shape:", selected_df.shape)
print("Unique county/puma pairs:", len(county_puma_df))
selected_df.head()

Filtered df shape: (549971, 5)


NameError: name 'county_puma_df' is not defined

In [5]:
# 5. Create county_group IDs
puma_to_group_id = {}
group_counter = 0
county_to_group = {}

# Assign groups for counties in multi-county PUMAs, and independent statuses
for county, puma in county_to_puma_group.items():
    if puma is not None:
        # County belongs to a multi-county PUMA; group it
        if puma not in puma_to_group_id:
            puma_to_group_id[puma] = f"county_group_{group_counter}"
            group_counter += 1
        county_to_group[county] = puma_to_group_id[puma]
    else:
        # County is either multi-PUMA or single-PUMA/single-county; treat both as independent
        county_to_group[county] = f"independent_county_{county}"

# 6. Add county_groups column to selected_df
selected_df['county_groups'] = selected_df['county'].map(county_to_group)

print(f"Multi-county PUMAs: {len(multi_county_pumas)}")
print(f"Independent counties (multi-PUMA + single-PUMA/single-county): {len(multi_puma_counties) + 196}")
print(f"Grouped counties (in multi-county PUMAs): {len([c for c in county_to_group if c not in multi_puma_counties and 'independent' not in county_to_group.get(c, '')])}")
print(f"Unique county groups: {selected_df['county_groups'].nunique()}")
print("\nCounty group summary:")
summary = selected_df.groupby(['county_groups']).size().sort_values(ascending=False)
print(f"Top 20 groups by building count:")
print(summary.head(20))

Multi-county PUMAs: 627
Independent counties (multi-PUMA + single-PUMA/single-county): 430
Grouped counties (in multi-county PUMAs): 2709
Unique county groups: 1046

County group summary:
Top 20 groups by building count:
county_groups
independent_county_G0600370    14320
independent_county_G1700310     8932
independent_county_G4802010     6923
independent_county_G0400130     6894
independent_county_G0600730     4872
independent_county_G0600590     4398
county_group_2                  4340
independent_county_G3600470     4192
independent_county_G4801130     4005
independent_county_G5300330     3620
independent_county_G3600610     3555
independent_county_G3200030     3547
independent_county_G3600810     3472
independent_county_G0600650     3364
independent_county_G1200110     3352
independent_county_G2601630     3350
county_group_211                3106
independent_county_G4804390     3037
independent_county_G0600710     2906
independent_county_G4800290     2788
dtype: int64


In [14]:
selected_df

,bldgid,county,state,puma,weight,county_groups
0,1,G0100590,AL,G01000100,253.903673,county_group_0
1,2,G0900090,CT,G09000903,253.903673,independent_county_G0900090
2,3,G2500170,MA,G25000502,253.903673,county_group_1
3,4,G4803030,TX,G48000502,253.903673,independent_county_G4803030
4,5,G0600650,CA,G06006510,253.903673,independent_county_G0600650
...,...,...,...,...,...,...
549966,549996,G1901690,IA,G19001300,253.903673,county_group_78
549967,549997,G1200190,FL,G12001900,253.903673,None
549968,549998,G4200790,PA,G42000802,253.903673,county_group_376
549969,549999,G0600590,CA,G06005910,253.903673,independent_county_G0600590


In [4]:
# Analyze counties with null county_groups
null_counties = selected_df[selected_df['county_groups'].isna()]['county'].unique()
print(f"Counties with null county_groups: {len(null_counties)}")
print(f"\nThese are 'single-PUMA counties': each contains one PUMA that is not multi-county")

# For each null county, show its PUMA(s)
null_county_pumas = selected_df[selected_df['county_groups'].isna()].groupby('county')['puma'].nunique().reset_index()
null_county_pumas.columns = ['county', 'num_pumas']

print(f"\nNull counties by number of PUMAs:")
print(null_county_pumas['num_pumas'].value_counts().sort_index())

# Verify: these counties should have PUMAs that are NOT in multi_county_pumas
example_null_county = null_counties[0]
example_puma = selected_df[selected_df['county'] == example_null_county]['puma'].iloc[0]
print(f"\nExample null county: {example_null_county}")
print(f"  PUMA: {example_puma}")
print(f"  Is this PUMA in multi_county_pumas? {example_puma in multi_county_pumas}")
print(f"  How many counties does this PUMA span? {counties_per_puma[counties_per_puma['puma'] == example_puma]['num_counties'].values[0]}")

print(f"\n--- Summary ---")
print(f"Single-PUMA counties (null): {len(null_counties)}")
print(f"Multi-PUMA independent counties: {len(multi_puma_counties)}")
print(f"Grouped counties (in county_groups): {len(selected_df[selected_df['county_groups'].notna()]['county'].unique())}")

Counties with null county_groups: 196

These are 'single-PUMA counties': each contains one PUMA that is not multi-county

Null counties by number of PUMAs:
num_pumas
1    196
Name: count, dtype: int64

Example null county: G1800950
  PUMA: G18001900
  Is this PUMA in multi_county_pumas? False
  How many counties does this PUMA span? 1

--- Summary ---
Single-PUMA counties (null): 196
Multi-PUMA independent counties: 234
Grouped counties (in county_groups): 2943


In [6]:
# Verify no null county_groups remain
null_count = selected_df['county_groups'].isna().sum()
print(f"Rows with null county_groups: {null_count}")
print(f"Total rows: {len(selected_df)}")
print(f"\nBreakdown of county_groups:")
print(f"  Independent counties: {selected_df['county_groups'].str.startswith('independent_county_').sum()}")
print(f"  Grouped counties: {selected_df['county_groups'].str.startswith('county_group_').sum()}")

Rows with null county_groups: 0
Total rows: 549971

Breakdown of county_groups:
  Independent counties: 343058
  Grouped counties: 206913


In [33]:
selected_df.to_csv("upgrade0_county_groups.csv", index=False)

In [8]:
# Diagnose Connecticut coverage and key format
if "selected_df" not in globals():
    tmp = df[["bldg_id", "in.county", "in.state", "in.puma", "weight"]].copy()
    tmp.columns = ["bldgid", "county", "state", "puma", "weight"]
    selected_df = tmp

ct_df = selected_df[selected_df["state"] == "CT"].copy()
print("CT rows:", len(ct_df))
print("Unique CT county keys:", ct_df["county"].nunique())
print("Unique CT PUMA keys:", ct_df["puma"].nunique())

print("\nSample CT county keys:")
print(sorted(ct_df["county"].unique())[:10])

print("\nSample CT PUMA keys:")
print(sorted(ct_df["puma"].unique())[:10])

# Check whether any CT rows have county_groups missing
if "county_groups" in selected_df.columns:
    ct_null = ct_df["county_groups"].isna().sum()
    print("\nCT rows with null county_groups:", int(ct_null))

# Show the normalized county key formats for map joins
ct_keys = ct_df[["county"]].drop_duplicates().copy()
ct_keys["county_stripped"] = ct_keys["county"].str.replace("^G", "", regex=True).str.rstrip("0")
ct_keys["county_fips5_guess"] = ct_keys["county"].str.replace("^G", "", regex=True).str.slice(0, 5)
print("\nCT county key conversion preview:")
print(ct_keys.head(10))

CT rows: 6132
Unique CT county keys: 8
Unique CT PUMA keys: 26

Sample CT county keys:
['G0900010', 'G0900030', 'G0900050', 'G0900070', 'G0900090', 'G0900110', 'G0900130', 'G0900150']

Sample CT PUMA keys:
['G09000100', 'G09000101', 'G09000102', 'G09000103', 'G09000104', 'G09000105', 'G09000300', 'G09000301', 'G09000302', 'G09000303']

CT rows with null county_groups: 0

CT county key conversion preview:
        county county_stripped county_fips5_guess
1     G0900090          090009              09000
16    G0900010          090001              09000
164   G0900030          090003              09000
376   G0900130          090013              09001
699   G0900070          090007              09000
1717  G0900110          090011              09001
2031  G0900150          090015              09001
4782  G0900050          090005              09000


In [74]:
# Quick fix: keep county_fips5 as a string key with leading zeros
# Handles three formats in `county`:
# 1) Code format (most states): G + state(2) + county4 + trailing 0, e.g. G0900010 -> 09001
# 2) Alaska text format: "AK, <borough/census area>" -> mapped via lookup
# 3) Hawaii text format: "HI, <county>" -> mapped via lookup

import csv

# Parse coded county values first
county_parts = selected_df["county"].str.extract(r"^G(?P<state>\d{2})(?P<county4>\d{4})0$")
selected_df["county_fips5"] = county_parts["state"] + county_parts["county4"].str[-3:]

# Alaska fallback for text county names
ak_name_to_fips = {
    "AK, Aleutians East Borough": "02013",
    "AK, Aleutians West Census Area": "02016",
    "AK, Anchorage Municipality": "02020",
    "AK, Bethel Census Area": "02050",
    "AK, Bristol Bay Borough": "02060",
    "AK, Denali Borough": "02068",
    "AK, Dillingham Census Area": "02070",
    "AK, Fairbanks North Star Borough": "02090",
    "AK, Haines Borough": "02100",
    "AK, Hoonah-Angoon Census Area": "02105",
    "AK, Juneau City and Borough": "02110",
    "AK, Kenai Peninsula Borough": "02122",
    "AK, Ketchikan Gateway Borough": "02130",
    "AK, Kodiak Island Borough": "02150",
    "AK, Kusilvak Census Area": "02158",
    "AK, Lake and Peninsula Borough": "02164",
    "AK, Matanuska-Susitna Borough": "02170",
    "AK, Nome Census Area": "02180",
    "AK, North Slope Borough": "02185",
    "AK, Northwest Arctic Borough": "02188",
    "AK, Petersburg Borough": "02195",
    "AK, Sitka City and Borough": "02220",
    "AK, Skagway Municipality": "02230",
    "AK, Southeast Fairbanks Census Area": "02240",
    "AK, Valdez-Cordova Census Area": "02261",
    "AK, Wrangell City and Borough": "02275",
    "AK, Yakutat City and Borough": "02282",
    "AK, Yukon-Koyukuk Census Area": "02290",
}

# Hawaii fallback for text county names
hi_name_to_fips = {
    "HI, Hawaii County": "15001",
    "HI, Honolulu County": "15003",
    "HI, Kalawao County": "15005",
    "HI, Kauai County": "15007",
    "HI, Maui County": "15009",
}

selected_df.loc[
    selected_df["county_fips5"].isna() & selected_df["county"].str.startswith("AK, ", na=False),
    "county_fips5",
] = selected_df.loc[
    selected_df["county_fips5"].isna() & selected_df["county"].str.startswith("AK, ", na=False),
    "county",
].map(ak_name_to_fips)

selected_df.loc[
    selected_df["county_fips5"].isna() & selected_df["county"].str.startswith("HI, ", na=False),
    "county_fips5",
] = selected_df.loc[
    selected_df["county_fips5"].isna() & selected_df["county"].str.startswith("HI, ", na=False),
    "county",
].map(hi_name_to_fips)

# Force county_fips5 to be a string and left-pad with zeros
selected_df["county_fips5"] = selected_df["county_fips5"].astype("string").str.zfill(5)

invalid_mask = selected_df["county_fips5"].isna() | (selected_df["county_fips5"].str.len() != 5)
invalid_fips = int(invalid_mask.sum())

print("Rows with invalid county_fips5:", invalid_fips)
print("Unique county_fips5:", selected_df["county_fips5"].nunique())

print("\nSample county_fips5 for CT/CA/AZ/CO/AK/AL:")
sample = selected_df[selected_df["state"].isin(["CT", "CA", "AZ", "CO", "AK", "AL"])][["state", "county", "county_fips5"]]
print(sample.drop_duplicates().sort_values(["state", "county"]).head(20))

# Export with quoting so county_fips5 stays text-like in many downstream tools
selected_df.to_csv("upgrade0_county_groups_with_fips.csv", index=False, quoting=csv.QUOTE_ALL)
print("\nWrote upgrade0_county_groups_with_fips.csv")

Rows with invalid county_fips5: 0
Unique county_fips5: 3139

Sample county_fips5 for CT/CA/AZ/CO/AK/AL:
       state                            county county_fips5
85571     AK        AK, Aleutians East Borough        02013
115369    AK    AK, Aleutians West Census Area        02016
1293      AK        AK, Anchorage Municipality        02020
8305      AK            AK, Bethel Census Area        02050
77964     AK           AK, Bristol Bay Borough        02060
22634     AK                AK, Denali Borough        02068
89642     AK        AK, Dillingham Census Area        02070
9096      AK  AK, Fairbanks North Star Borough        02090
22713     AK                AK, Haines Borough        02100
73149     AK     AK, Hoonah-Angoon Census Area        02105
4507      AK       AK, Juneau City and Borough        02110
2896      AK       AK, Kenai Peninsula Borough        02122
4672      AK     AK, Ketchikan Gateway Borough        02130
411       AK         AK, Kodiak Island Borough        02

In [14]:
# Diagnose invalid county_fips5 patterns for selected states
states_to_check = ["CT", "CA", "AZ", "CO", "AK", "AL"]

check_df = selected_df[selected_df["state"].isin(states_to_check)].copy()
check_df["county_digits"] = check_df["county"].str.replace(r"^G", "", regex=True)
check_df["county_digits_len"] = check_df["county_digits"].str.len()

invalid_df = check_df[
    check_df["county_fips5"].isna() | (check_df["county_fips5"].str.len() != 5)
].copy()

print("Invalid rows across target states:", len(invalid_df))
print("\nInvalid rows by state:")
print(invalid_df.groupby("state").size().sort_values(ascending=False))

print("\nDigit-length distribution for county codes in target states:")
print(check_df.groupby(["state", "county_digits_len"]).size())

print("\nSample invalid county codes:")
print(invalid_df[["state", "county"]].drop_duplicates().sort_values(["state", "county"]).head(50))

Invalid rows across target states: 1255

Invalid rows by state:
state
AK    1255
dtype: int64

Digit-length distribution for county codes in target states:
state  county_digits_len
AK     18                      14
       20                      17
       22                      32
       23                      14
       24                      11
       25                      22
       26                     501
       27                     180
       28                      12
       29                     226
       30                      39
       32                     171
       35                      16
AL     7                     9065
AZ     7                    11952
CA     7                    57075
CO     7                     9374
CT     7                     6132
dtype: int64

Sample invalid county codes:
       state                               county
85571     AK           AK, Aleutians East Borough
115369    AK       AK, Aleutians West Census Area
1293      AK  

In [16]:
# Inspect remaining invalid county_fips5 values
remaining_invalid = selected_df[selected_df["county_fips5"].isna() | (selected_df["county_fips5"].str.len() != 5)]
print("Remaining invalid rows:", len(remaining_invalid))
print(remaining_invalid.groupby("state").size())
print("\nSample remaining invalid county values:")
print(remaining_invalid[["state", "county"]].drop_duplicates().sort_values(["state", "county"]).head(50))

Remaining invalid rows: 2163
state
HI    2163
dtype: int64

Sample remaining invalid county values:
     state               county
69      HI    HI, Hawaii County
6       HI  HI, Honolulu County
6003    HI     HI, Kauai County
737     HI      HI, Maui County


In [23]:
# Explicit sample for CT/CA/AZ/CO/AL after county_fips5 string fix
for st in ["CT", "CA", "AZ", "CO", "AL"]:
    sample = (
        selected_df[selected_df["state"] == st][["state", "county", "county_fips5"]]
        .drop_duplicates()
        .sort_values("county")
        .head(5)
    )
    print(f"\n{st} sample:")
    print(sample)


CT sample:
     state    county county_fips5
16      CT  G0900010        09001
164     CT  G0900030        09003
4782    CT  G0900050        09005
699     CT  G0900070        09007
1       CT  G0900090        09009

CA sample:
      state    county county_fips5
49       CA  G0600010        06001
48160    CA  G0600030        06003
8286     CA  G0600050        06005
2379     CA  G0600070        06007
2308     CA  G0600090        06009

AZ sample:
      state    county county_fips5
5978     AZ  G0400010        04001
1620     AZ  G0400030        04003
5261     AZ  G0400050        04005
4739     AZ  G0400070        04007
11727    AZ  G0400090        04009

CO sample:
      state    county county_fips5
246      CO  G0800010        08001
41643    CO  G0800030        08003
51       CO  G0800050        08005
47741    CO  G0800070        08007
12279    CO  G0800090        08009

AL sample:
      state    county county_fips5
638      AL  G0100010        01001
221      AL  G0100030        01003
3

In [75]:
# Rebuild county_groups from scratch and ensure CT counties are assigned
base = selected_df[["county", "state", "puma", "weight"]].copy()

# Weight by county-puma pair
weight_by_cp = base.groupby(["county", "puma"], as_index=False)["weight"].sum()

# Multi-county PUMAs
counties_per_puma = base.groupby("puma", as_index=False)["county"].nunique().rename(columns={"county": "num_counties"})
multi_county_pumas = set(counties_per_puma.loc[counties_per_puma["num_counties"] > 1, "puma"])

# Counties where all PUMAs are county-local (independent by design)
pumas_per_county = base.groupby("county", as_index=False)["puma"].nunique().rename(columns={"puma": "num_pumas"})
multi_puma_counties = set()
for county in pumas_per_county.loc[pumas_per_county["num_pumas"] > 1, "county"]:
    c_pumas = set(base.loc[base["county"] == county, "puma"].unique())
    if all(p not in multi_county_pumas for p in c_pumas):
        multi_puma_counties.add(county)

# Choose dominant multi-county PUMA per county (if county appears in multiple such PUMAs)
county_to_puma_group = {}
for county in base["county"].dropna().unique():
    if county in multi_puma_counties:
        county_to_puma_group[county] = None
        continue

    county_pumas = set(base.loc[base["county"] == county, "puma"].unique())
    candidates = [p for p in county_pumas if p in multi_county_pumas]

    if len(candidates) == 0:
        county_to_puma_group[county] = None
    elif len(candidates) == 1:
        county_to_puma_group[county] = candidates[0]
    else:
        candidate_weights = {}
        for p in candidates:
            w = weight_by_cp.loc[(weight_by_cp["county"] == county) & (weight_by_cp["puma"] == p), "weight"].sum()
            candidate_weights[p] = w
        county_to_puma_group[county] = max(candidate_weights, key=candidate_weights.get)

# Build initial county group labels
puma_to_group_id = {}
group_counter = 0
county_to_group = {}

for county, puma in county_to_puma_group.items():
    if puma is not None:
        if puma not in puma_to_group_id:
            puma_to_group_id[puma] = f"county_group_{group_counter}"
            group_counter += 1
        county_to_group[county] = puma_to_group_id[puma]
    else:
        county_to_group[county] = f"independent_county_{county}"

selected_df["county_groups"] = selected_df["county"].map(county_to_group)

# CT safeguard: any CT county left unassigned gets independent_county_<county>
ct_missing_mask = (selected_df["state"] == "CT") & (selected_df["county_groups"].isna() | (selected_df["county_groups"].astype(str).str.len() == 0))
selected_df.loc[ct_missing_mask, "county_groups"] = "independent_county_" + selected_df.loc[ct_missing_mask, "county"].astype(str)

# Convert singleton county_group_* labels to independent_county_* labels
county_group_size = (
    selected_df[["county", "county_groups"]]
    .drop_duplicates()
    .groupby("county_groups", as_index=False)["county"]
    .nunique()
    .rename(columns={"county": "num_counties"})
)

singleton_groups = set(
    county_group_size.loc[
        county_group_size["county_groups"].astype(str).str.startswith("county_group_")
        & (county_group_size["num_counties"] == 1),
        "county_groups",
    ]
)

singleton_mask = selected_df["county_groups"].isin(singleton_groups)
selected_df.loc[singleton_mask, "county_groups"] = "independent_county_" + selected_df.loc[singleton_mask, "county"].astype(str)

ct_missing_counties = selected_df.loc[(selected_df["state"] == "CT") & selected_df["county_groups"].isna(), "county"].drop_duplicates()
print("CT rows:", int((selected_df["state"] == "CT").sum()))
print("CT rows with missing county_groups:", int(((selected_df["state"] == "CT") & selected_df["county_groups"].isna()).sum()))
print("CT counties with missing county_groups:", len(ct_missing_counties))
print("Singleton county_group labels converted:", len(singleton_groups))
print("Total unique county_groups:", selected_df["county_groups"].nunique())

CT rows: 6132
CT rows with missing county_groups: 0
CT counties with missing county_groups: 0
Singleton county_group labels converted: 37
Total unique county_groups: 1046


In [30]:
# Diagnose singleton county group behavior (example: Elbert County, CO)
co_elbert = selected_df[(selected_df["state"] == "CO") & (selected_df["county"] == "G0800390")].copy()

print("Elbert rows:", len(co_elbert))
print("Elbert county_groups:", co_elbert["county_groups"].drop_duplicates().tolist())
print("Elbert PUMAs:", sorted(co_elbert["puma"].dropna().unique().tolist()))

# Weight by PUMA for Elbert
elbert_w = co_elbert.groupby("puma", as_index=False)["weight"].sum().sort_values("weight", ascending=False)
print("\nElbert weight by PUMA:")
print(elbert_w)

# For each Elbert PUMA, show all counties using that PUMA and their assigned groups
for p in elbert_w["puma"].tolist():
    sub = selected_df[selected_df["puma"] == p][["county", "state", "county_groups", "weight"]].copy()
    by_county = sub.groupby(["state", "county", "county_groups"], as_index=False)["weight"].sum().sort_values("weight", ascending=False)
    print(f"\nPUMA {p} counties and assigned groups:")
    print(by_county.head(20))

# Count counties per county_group to identify singletons
county_group_size = (
    selected_df[["county", "county_groups"]]
    .drop_duplicates()
    .groupby("county_groups", as_index=False)["county"]
    .nunique()
    .rename(columns={"county": "num_counties"})
)

elbert_group = co_elbert["county_groups"].iloc[0] if len(co_elbert) else None
if elbert_group is not None:
    group_row = county_group_size[county_group_size["county_groups"] == elbert_group]
    print("\nElbert group size:")
    print(group_row)

print("\nExample singleton groups (num_counties == 1):")
print(county_group_size[county_group_size["num_counties"] == 1].head(20))

Elbert rows: 37
Elbert county_groups: ['independent_county_G0800390']
Elbert PUMAs: ['G08000100', 'G08000823']

Elbert weight by PUMA:
        puma       weight
1  G08000823  7617.110182
0  G08000100  1777.325709

PUMA G08000823 counties and assigned groups:
  state    county                county_groups        weight
0    CO  G0800350  independent_county_G0800350  39355.069273
1    CO  G0800390  independent_county_G0800390   7617.110182

PUMA G08000100 counties and assigned groups:
   state    county                county_groups        weight
8     CO  G0800870              county_group_45  11933.472618
7     CO  G0800750              county_group_45   9394.435891
13    CO  G0801250              county_group_45   4570.266109
5     CO  G0800630              county_group_45   3554.651418
11    CO  G0801210              county_group_45   2539.036727
0     CO  G0800110              county_group_45   2285.133055
12    CO  G0801230             county_group_144   2285.133055
6     CO  G08007

In [32]:
# Quick check: Elbert County, CO current label
print(
    selected_df.loc[
        (selected_df["state"] == "CO") & (selected_df["county"] == "G0800390"),
        "county_groups",
    ].drop_duplicates().tolist()
)

['independent_county_G0800390']


In [78]:
# Add one dummy row per missing county_fips5 using 2010 PUMA lookup.
# PUMA format must match dataset convention: G + state(2) + puma(6).

# 1) Complete county reference (50 states + DC)
state_to_fips = {
    "AL": "01", "AK": "02", "AZ": "04", "AR": "05", "CA": "06", "CO": "08", "CT": "09", "DE": "10",
    "DC": "11", "FL": "12", "GA": "13", "HI": "15", "ID": "16", "IL": "17", "IN": "18", "IA": "19",
    "KS": "20", "KY": "21", "LA": "22", "ME": "23", "MD": "24", "MA": "25", "MI": "26", "MN": "27",
    "MS": "28", "MO": "29", "MT": "30", "NE": "31", "NV": "32", "NH": "33", "NJ": "34", "NM": "35",
    "NY": "36", "NC": "37", "ND": "38", "OH": "39", "OK": "40", "OR": "41", "PA": "42", "RI": "44",
    "SC": "45", "SD": "46", "TN": "47", "TX": "48", "UT": "49", "VT": "50", "VA": "51", "WA": "53",
    "WV": "54", "WI": "55", "WY": "56",
}
fips_to_state = {v: k for k, v in state_to_fips.items()}

county_ref_url = "https://www2.census.gov/geo/docs/reference/codes2020/national_county2020.txt"
county_ref = pd.read_csv(county_ref_url, sep="|", dtype=str)
county_ref = county_ref[county_ref["STATEFP"].isin(set(state_to_fips.values()))].copy()
county_ref["county_fips5"] = county_ref["STATEFP"] + county_ref["COUNTYFP"]
county_ref["state"] = county_ref["STATEFP"].map(fips_to_state)
county_ref["county"] = "G" + county_ref["STATEFP"] + county_ref["COUNTYFP"] + "0"

# Add county_name string for all existing rows using county_fips5 lookup
county_name_lookup = county_ref.set_index("county_fips5")["COUNTYNAME"].to_dict()
selected_df["county_name"] = selected_df["county_fips5"].map(county_name_lookup).astype("string")

# 2) Missing county_fips5 values relative to selected_df
existing_fips5 = set(selected_df["county_fips5"].dropna().astype(str).unique())
missing_ref = county_ref[~county_ref["county_fips5"].isin(existing_fips5)].copy()

# 3) 2010 tract -> county -> dominant PUMA lookup
tract_puma_url = "https://www2.census.gov/geo/docs/maps-data/data/rel/2010_Census_Tract_to_2010_PUMA.txt"
tract_puma = pd.read_csv(tract_puma_url, dtype=str)
tract_puma["county_fips5"] = tract_puma["STATEFP"] + tract_puma["COUNTYFP"]
tract_puma["puma_g"] = "G" + tract_puma["STATEFP"] + tract_puma["PUMA5CE"].str.zfill(6)

county_puma_counts = (
    tract_puma.groupby(["county_fips5", "puma_g"], as_index=False)
    .size()
    .rename(columns={"size": "tract_count"})
)
county_puma_primary = county_puma_counts.sort_values(
    ["county_fips5", "tract_count", "puma_g"], ascending=[True, False, True]
).drop_duplicates("county_fips5")
county_to_puma = dict(zip(county_puma_primary["county_fips5"], county_puma_primary["puma_g"]))

# 4) Existing PUMA -> county_group mapping (weighted majority)
puma_group_weights = (
    selected_df.dropna(subset=["puma", "county_groups"])
    .groupby(["puma", "county_groups"], as_index=False)["weight"]
    .sum()
)
puma_group_majority = puma_group_weights.sort_values(
    ["puma", "weight", "county_groups"], ascending=[True, False, True]
).drop_duplicates("puma")
puma_to_group = dict(zip(puma_group_majority["puma"], puma_group_majority["county_groups"]))

# 5) Build one dummy row per missing county_fips5 and populate all selected_df columns
dummy_rows = []
for _, r in missing_ref.iterrows():
    county_fips5 = r["county_fips5"]
    county_code = r["county"]
    state = r["state"]

    puma_val = county_to_puma.get(county_fips5)
    if puma_val in puma_to_group:
        county_group = puma_to_group[puma_val]
    else:
        county_group = f"independent_county_{county_code}"

    base_row = {
        "bldgid": f"dummy_{county_fips5}",
        "county": county_code,
        "county_name": r["COUNTYNAME"],
        "state": state,
        "puma": puma_val,
        "weight": 0.0,
        "county_groups": county_group,
        "county_fips5": county_fips5,
    }
    full_row = {col: base_row[col] if col in base_row else pd.NA for col in selected_df.columns}
    dummy_rows.append(full_row)

dummy_df = pd.DataFrame(dummy_rows, columns=selected_df.columns)
before_rows = len(selected_df)
selected_df = pd.concat([selected_df, dummy_df], ignore_index=True)

print(f"County ref rows (50 states + DC): {len(county_ref):,}")
print(f"Existing county_fips5: {len(existing_fips5):,}")
print(f"Missing county_fips5 identified: {len(missing_ref):,}")
print(f"Dummy rows added: {len(dummy_df):,}")
print(f"Rows before: {before_rows:,}")
print(f"Rows after:  {len(selected_df):,}")

loving = selected_df[selected_df["county_fips5"].astype(str) == "48301"][
    ["bldgid", "state", "county", "county_name", "puma", "county_groups", "county_fips5", "weight"]
]
print("\nLoving County (48301):")
print(loving.head(5))

County ref rows (50 states + DC): 3,143
Existing county_fips5: 3,144
Missing county_fips5 identified: 0
Dummy rows added: 0
Rows before: 549,976
Rows after:  549,976

Loving County (48301):
             bldgid state   county    county_name       puma  \
549975  dummy_48301    TX  G483010  Loving County  G48003200   

           county_groups county_fips5  weight  
549975  county_group_490        48301     0.0  


/var/folders/71/8g_0bty57c79j0q7qrbx6g68w51x81/T/ipykernel_61523/3876008446.py:86: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  selected_df = pd.concat([selected_df, dummy_df], ignore_index=True)


In [79]:
# Export
selected_df.to_csv("upgrade0_county_groups.csv", index=False)
selected_df.to_csv("upgrade0_county_groups_with_fips.csv", index=False, quoting=csv.QUOTE_ALL)
print("✓ Exported CSVs with dummy rows")


✓ Exported CSVs with dummy rows


In [71]:
# show rows in TX
selected_df[(selected_df["state"] == "TX") & (selected_df["county_fips5"] == "48301")]

,bldgid,county,state,puma,weight,county_fips5,county_groups
549975,dummy_48301,G483010,TX,G4803200,0.0,48301,independent_county_G483010


In [53]:
import pandas as pd

tract_puma_url = "https://www2.census.gov/geo/docs/maps-data/data/rel/2010_Census_Tract_to_2010_PUMA.txt"
tract_puma = pd.read_csv(tract_puma_url, dtype=str)
print("tract_puma columns:", tract_puma.columns.tolist())
print("tract_puma rows:", len(tract_puma))
print(tract_puma.head(3))

print("\nselected_df puma sample:")
print(selected_df["puma"].dropna().astype(str).head(10).tolist())
print("unique puma count:", selected_df["puma"].nunique())

tract_puma columns: ['STATEFP', 'COUNTYFP', 'TRACTCE', 'PUMA5CE']
tract_puma rows: 74091
  STATEFP COUNTYFP TRACTCE PUMA5CE
0      01      001  020100   02100
1      01      001  020200   02100
2      01      001  020300   02100

selected_df puma sample:
['G01000100', 'G09000903', 'G25000502', 'G48000502', 'G06006510', 'G12008610', 'HI, 00308', 'G47001603', 'G22002200', 'G12008615']
unique puma count: 2351
